# EDSS employment school-year mart validation

## tl;dr

The restricted 2010–2022 employment view aggregates from 7,277,987 records to 7,058 unique school-year keys and joins the 24,044-row core without expansion. All 537 school metric vectors in 2022 exactly match 2021, so 2022 is marked ineligible for time comparison; reported further-study counts are all zero in 2016–2019.

## Context & Methods

This notebook validates the materialized school-year employment table, its left join to the school-year core, the committed dictionary, and the build audit. It uses aggregate counts only and does not display person-level fields.

### Key Assumptions

- The source cohort is `analysis.employment_legacy_2010_2022`; 2023–2024 remains outside the OpenID longitudinal scope.
- The 11 selected source fields are binary record-level reported indicators and are summed by `(_panel_year, 개방ID)`.
- 2022 values are preserved but not treated as an independent time observation.
- No official employment rate is derived because the denominator and exclusion fields are not stable enough across years.

In [1]:
from pathlib import Path
import csv
import json
import duckdb

repo_root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
database_path = repo_root / 'data/processed/edss/restricted/edss_all.duckdb'
audit_path = repo_root / 'data/metadata/edss_duckdb_build.json'
dictionary_path = repo_root / 'data/metadata/edss_employment_school_year_data_dictionary.csv'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
dictionary = list(csv.DictReader(dictionary_path.open(encoding='utf-8-sig', newline='')))
mart_audit = audit['employment_school_year_mart']
con = duckdb.connect(str(database_path), read_only=True)
print({'duckdb_version': duckdb.__version__, 'table': mart_audit['table'], 'database_sha256': audit['database']['sha256']})

{'duckdb_version': '1.4.1', 'table': 'analysis.employment_school_year_2010_2022', 'database_sha256': '3d200cfb90b6118adfe9adf5d1f1405b9cc32acc01ef6b41332f66250ceee53a'}


## Data

The aggregate table exposes only year, OpenID, three quality fields, source-record count, and 11 reported count metrics.

In [2]:
schema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'analysis'
      AND table_name = 'employment_school_year_2010_2022'
    ORDER BY ordinal_position
""").fetchall()
expected_schema = [(row['column_name'], row['data_type']) for row in dictionary]
assert schema == expected_schema, (schema, expected_schema)
assert len(schema) == 17
print({'columns': len(schema), 'dictionary_rows': len(dictionary)})

{'columns': 17, 'dictionary_rows': 17}


## Results

The mart must retain every source record in exactly one school-year aggregate, use a unique key, and join the core without adding or dropping core rows.

In [3]:
mart_stats = con.execute("""
    SELECT count(*), count(DISTINCT (_panel_year, 개방ID)),
           count(*) FILTER (WHERE coalesce(_panel_year, '') = '' OR coalesce(개방ID, '') = ''),
           min(_panel_year), max(_panel_year), sum(source_record_count),
           count(*) FILTER (WHERE employment_time_comparison_eligible = false)
    FROM analysis.employment_school_year_2010_2022
""").fetchone()
joined_stats = con.execute("""
    SELECT count(*), count(DISTINCT (_panel_year, 개방ID)),
           count(*) FILTER (WHERE _employment_exists = 'true'),
           count(*) FILTER (WHERE _employment_exists = 'false')
    FROM analysis.school_year_core_with_employment_2010_2022
""").fetchone()
assert mart_stats == (7058, 7058, 0, '2010', '2022', 7277987, 537)
assert joined_stats == (24044, 24044, 7058, 16986)
assert mart_audit['orphan_employment_key_count'] == 0
assert mart_audit['join_expansion_count'] == 0
print({'mart': mart_stats, 'core_join': joined_stats})

{'mart': (7058, 7058, 0, '2010', '2022', 7277987, 537), 'core_join': (24044, 24044, 7058, 16986)}


Every selected source value must be present, binary, and nonnegative; source and mart sums must reconcile. Cross-field subset checks must also pass.

In [4]:
metric_validation = mart_audit['metric_validation']
assert len(metric_validation) == 11
assert all(item['nonblank_row_count'] == 7277987 for item in metric_validation.values())
assert all(item['invalid_nonblank_row_count'] == 0 for item in metric_validation.values())
assert all(item['negative_row_count'] == 0 for item in metric_validation.values())
assert all(item['nonbinary_row_count'] == 0 for item in metric_validation.values())
assert all(item['sum_reconciles'] is True for item in metric_validation.values())
assert all(value == 0 for value in mart_audit['subset_violations'].values())
print({'metrics': len(metric_validation), 'subset_checks': len(mart_audit['subset_violations'])})

{'metrics': 11, 'subset_checks': 11}


The anomaly checks compare complete school-level metric vectors for 2021 and 2022 and identify years whose reported further-study total is zero.

In [5]:
quality = mart_audit['quality_findings']
duplicate = quality['duplicate_year_comparison']
assert duplicate == {
    'base_year': '2021',
    'comparison_year': '2022',
    'base_key_count': 537,
    'comparison_key_count': 537,
    'shared_key_count': 537,
    'exact_metric_vector_match_key_count': 537,
    'exact_duplicate_detected': True,
    'comparison_eligible': False,
}
assert quality['all_zero_reported_further_study_years'] == ['2016', '2017', '2018', '2019']
assert quality['official_employment_rate_derived'] is False
print({'duplicate_year': duplicate, 'zero_further_study_years': quality['all_zero_reported_further_study_years']})

{'duplicate_year': {'base_year': '2021', 'comparison_year': '2022', 'base_key_count': 537, 'comparison_key_count': 537, 'shared_key_count': 537, 'exact_metric_vector_match_key_count': 537, 'exact_duplicate_detected': True, 'comparison_eligible': False}, 'zero_further_study_years': ['2016', '2017', '2018', '2019']}


Annual aggregate output is bounded to 13 rows and makes the two quality statuses visible.

In [6]:
year_summary = con.execute("""
    SELECT _panel_year, sum(source_record_count),
           sum(reported_employed_count), sum(reported_further_study_count),
           bool_and(employment_time_comparison_eligible),
           min(further_study_quality_status)
    FROM analysis.employment_school_year_2010_2022
    GROUP BY _panel_year
    ORDER BY _panel_year
""").fetchall()
assert len(year_summary) == 13
assert year_summary[-2][1:4] == year_summary[-1][1:4]
for row in year_summary:
    print(row)

('2010', 539071, 267003, 36105, True, 'as_reported')
('2011', 559000, 292028, 35705, True, 'as_reported')
('2012', 566374, 294884, 37675, True, 'as_reported')
('2013', 555141, 284660, 39641, True, 'as_reported')
('2014', 557236, 281663, 39763, True, 'as_reported')
('2015', 557234, 302280, 34618, True, 'as_reported')
('2016', 576023, 315412, 0, True, 'all_zero_source_field')
('2017', 580695, 318438, 0, True, 'all_zero_source_field')
('2018', 574009, 305263, 0, True, 'all_zero_source_field')
('2019', 555808, 299883, 0, True, 'all_zero_source_field')
('2020', 550354, 291210, 30294, True, 'as_reported')
('2021', 553521, 283694, 31150, True, 'as_reported')
('2022', 553521, 283694, 31150, False, 'as_reported')


In [7]:
con.close()
print('employment school-year mart validation complete')

employment school-year mart validation complete


## Takeaways

- All 7,277,987 restricted records reconcile to 7,058 unique school-year aggregates.
- The core join remains 24,044 unique rows; 16,986 rows without employment data retain null employment metrics.
- 2022 is preserved but excluded from time comparison because all 537 school metric vectors duplicate 2021.
- Reported further-study counts for 2016–2019 are structurally all zero and must not be interpreted as observed zero participation.
- No official employment rate is derived from these fields.